# Building CodAdapt

## Goal

Build CodAdapt from scratch to understand how the algorithm works step by step.

CodAdapt is an experimental tabular machine learning library I am currently developing, available on GitHub and PyPI.

This project rebuilds the core ideas from the simplest possible version and gradually introduces bucketization, coded memory, collisions, and adaptive codes.

---

## Version 1

Build the simplest possible bucket-memory model using one numerical feature.

The model will:

- divide a numerical feature into buckets
- learn one prediction for each bucket
- store the predictions in a small memory
- use the memory to make predictions

In [1]:
import numpy as np
import pandas as pd

In [2]:
PATH = "/kaggle/input/competitions/titanic/"

train = pd.read_csv(PATH + "train.csv", index_col="PassengerId")
test  = pd.read_csv(PATH + "test.csv", index_col="PassengerId")

## Data preparation

We use `Age` as the only input feature and `Survived` as the target.

Missing ages are filled using the median learned from the training data.

In [3]:
target = "Survived"

age_median = train["Age"].median()

x = train["Age"].fillna(age_median).to_numpy()
y = train[target].to_numpy()

## Building the model

We first divide the continuous `Age` values into four buckets.

Each bucket will point to one value stored in memory.

In [4]:
cuts = np.quantile(x, [0.25, 0.50, 0.75])

bucket = np.digitize(x, cuts)

In [5]:
cuts

array([22., 28., 35.])

In [6]:
bucket[:10]

array([1, 3, 1, 3, 3, 2, 3, 0, 1, 0])

Each memory cell learns the average target value of the samples assigned to that bucket.

In [7]:
memory = np.zeros(4)

In [8]:
for b in range(4):
    memory[b] = y[bucket == b].mean()

In [9]:
memory

array([0.42647059, 0.40601504, 0.3354232 , 0.4       ])

A prediction is obtained by using the bucket as an address into memory.

In [10]:
preds = memory[bucket]

In [11]:
preds[:10]

array([0.40601504, 0.4       , 0.40601504, 0.4       , 0.4       ,
       0.3354232 , 0.4       , 0.42647059, 0.40601504, 0.42647059])

## Prediction and submission

The test ages are converted into buckets using the same cut points learned from the training data.

Each bucket directly selects its prediction from memory.

In [12]:
x_test = test["Age"].fillna(age_median).to_numpy()

bucket_test = np.digitize(x_test, cuts)

probabilities = memory[bucket_test]
preds = (probabilities >= 0.5).astype(int)

In [13]:
submission = pd.DataFrame({
    "PassengerId": test.index,
    "Survived": preds
})

submission.to_csv("submission_v1.csv", index=False)

submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0


## Notes

- This first version focuses only on the basic memory mechanism behind CodAdapt.
- `Age` is used as the only input feature.
- Missing `Age` values are filled using the median learned from the training data.
- Continuous values are discretized into four buckets using quantiles.
- Each bucket stores the average target value observed during training.
- Predictions are obtained by reading the value stored in memory for each bucket.
- The same bucket boundaries learned from the training data are reused for the test data.
- The model does not yet use multiple features, feature interactions, coded memory, collisions, shared multi-resolution encoding, or adaptive codes.
- The final probabilities are converted to `0` or `1` using a threshold of `0.5` for the Titanic submission.
- This notebook is a simplified educational reconstruction of CodAdapt, not the full library implementation.
- CodAdapt is an experimental tabular machine learning library I am currently developing, available on GitHub and PyPI.

Future versions will introduce new components and gradually evolve the architecture.